In [ ]:
import numpy as np


def is_effectively_shielded(
    missile_pos,
    target_points,
    cloud_center,
    explosion_time,
    current_time,
    smoke_radius=10,
    last_time=20,
):
    """
    判断导弹是否被有效遮蔽

    参数:
    missile_pos -- 导弹当前位置坐标 (x,y,z)
    target_points -- 目标上的采样点列表
    cloud_center -- 烟幕云团中心位置 (x,y,z)
    explosion_time -- 烟幕弹爆炸时间
    current_time -- 当前时间
    smoke_radius -- 烟幕云团半径,10m
    last_time -- 烟幕有效持续时间,20s

    返回:
    bool -- 是否被有效遮蔽
    """
    # 时间检查
    if current_time - explosion_time > last_time:
        return False

    # 位置检查
    for target_point in target_points:
        # 导弹到目标点的视线向量
        sight_vector = np.array(target_point) - np.array(missile_pos)
        sight_length = np.linalg.norm(sight_vector)
        if sight_length < 1e-6:  # 避免除零
            return True

        # 云团中心到视线的投影比例
        mc_vector = np.array(cloud_center) - np.array(missile_pos)
        t = np.dot(mc_vector, sight_vector) / (sight_length**2)

        # 云团中心到视线的垂直距离
        distance = np.linalg.norm(mc_vector - t * sight_vector)

        # 云团是否在导弹和目标之间
        in_between = 0 <= t <= 1

        # 如果视线未被遮蔽或云团不在中间，则整体未被遮蔽
        if distance >= smoke_radius:
            # 如果云团离视线太远，肯定不遮蔽
            return False
        elif not in_between:
            # 如果云团不在导弹和目标之间，检查是否足够接近目标
            if t > 1:  # 云团在目标"后方"
                distance_to_target = np.linalg.norm(
                    np.array(cloud_center) - np.array(target_point)
                )
                return distance_to_target < smoke_radius
            else:  # 云团在导弹"后方"(t<0)
                distance_to_missile = np.linalg.norm(
                    np.array(cloud_center) - np.array(missile_pos))
                return distance_to_missile < smoke_radius            
    return True

def is_effectively_shielded_by_multiple_clouds(
    missile_pos,
    target_points,
    cloud_centers,
    explosion_times,
    current_time,
    smoke_radius=10,
    last_time=20,
):
    """
    判断导弹是否被多个烟幕云团有效遮蔽
    
    参数:
    missile_pos -- 导弹当前位置坐标 (x,y,z)
    target_points -- 目标上的采样点列表
    cloud_centers -- 多个烟幕云团中心位置列表，每个元素为 (x,y,z) 或 None
    explosion_times -- 多个烟幕弹爆炸时间列表
    current_time -- 当前时间
    smoke_radius -- 烟幕云团半径,10m
    last_time -- 烟幕有效持续时间,20s
    
    返回:
    bool -- 是否被有效遮蔽
    """
    # 筛选有效的云团(已爆炸且未超过持续时间)
    active_clouds = []
    for cloud_center, explosion_time in zip(cloud_centers, explosion_times):
        if cloud_center is not None and current_time - explosion_time <= last_time:
            active_clouds.append(cloud_center)
    
    # 如果没有有效云团，直接返回False
    if not active_clouds:
        return False
    
    # 检查每个目标点
    for target_point in target_points:
        # 导弹到目标点的视线向量
        sight_vector = np.array(target_point) - np.array(missile_pos)
        sight_length = np.linalg.norm(sight_vector)
        
        # 如果导弹已经到达目标点，视为被遮蔽
        if sight_length < 1e-6:
            return True
        
        # 检查是否至少有一个云团遮蔽了视线
        line_shielded = False
        
        for cloud_center in active_clouds:
            # 云团中心到视线的投影比例
            mc_vector = np.array(cloud_center) - np.array(missile_pos)
            t = np.dot(mc_vector, sight_vector) / (sight_length**2)
            
            # 云团中心到视线的垂直距离
            distance = np.linalg.norm(mc_vector - t * sight_vector)
            
            # 云团是否在导弹和目标之间
            in_between = 0 <= t <= 1
            
            # 判断该云团是否遮蔽了视线
            if distance < smoke_radius and in_between:
                line_shielded = True
                break
            elif distance < smoke_radius:
                # 如果云团不在导弹和目标之间，检查是否足够接近目标或导弹
                if t > 1:  # 云团在目标"后方"
                    distance_to_target = np.linalg.norm(
                        np.array(cloud_center) - np.array(target_point)
                    )
                    if distance_to_target < smoke_radius:
                        line_shielded = True
                        break
                else:  # 云团在导弹"后方"(t<0)
                    distance_to_missile = np.linalg.norm(
                        np.array(cloud_center) - np.array(missile_pos)
                    )
                    if distance_to_missile < smoke_radius:
                        line_shielded = True
                        break
        
        # 如果所有云团都未能遮蔽该视线，则整体未被遮蔽
        if not line_shielded:
            return False
    
    # 所有视线都被遮蔽，则整体被遮蔽
    return True

def calculate_missile_position(initial_pos, time, velocity=300, target_pos=(0, 0, 0)):
    """
    计算导弹在给定时间的位置

    参数:
    initial_pos -- 导弹初始位置 (x,y,z)
    time -- 时间(秒)
    velocity -- 导弹速度(米/秒),默认300
    target_pos -- 导弹目标位置(假目标)，默认(0,0,0)

    返回:
    tuple -- 导弹当前位置 (x,y,z)
    """
    initial_pos = np.array(initial_pos)
    target_pos = np.array(target_pos)

    direction = target_pos - initial_pos
    direction_unit = direction / np.linalg.norm(direction)
    velocity_vector = direction_unit * velocity

    new_pos = initial_pos + velocity_vector * time
    return tuple(new_pos)


def calculate_plane_position(initial_pos, direction, speed, time):
    """
    计算飞机在给定时间的位置

    参数:
    initial_pos -- 飞机初始位置 (x,y,z)
    direction -- 飞行方向向量
    speed -- 飞行速度(米/秒)
    time -- 时间(秒)

    返回:
    tuple -- 飞机当前位置 (x,y,z)
    """
    initial_pos = np.array(initial_pos)
    direction = np.array(direction)

    direction_unit = direction / np.linalg.norm(direction)
    velocity_vector = direction_unit * speed

    new_pos = initial_pos + velocity_vector * time
    return tuple(new_pos)


def calculate_bomb_position(
    plane_initial_pos,
    plane_direction,
    plane_speed,
    release_time,
    fall_time,
    gravity=9.8,
):
    """
    计算烟幕弹轨迹和爆炸位置

    参数:
    plane_initial_pos -- 无人机初始位置 (x,y,z)
    plane_direction -- 无人机飞行方向
    plane_speed -- 无人机速度(米/秒)
    release_time -- 投放时间(秒)
    fall_time -- 烟幕弹下落时间(秒)
    gravity -- 重力加速度,默认9.8

    返回:
    tuple -- 爆炸位置 (x,y,z)
    """
    # 计算投放位置
    release_position = calculate_plane_position(
        plane_initial_pos, plane_direction, plane_speed, release_time
    )

    # 计算水平位移
    plane_direction = np.array(plane_direction)
    direction_unit = plane_direction / np.linalg.norm(plane_direction)
    velocity_vector = direction_unit * plane_speed

    # 计算爆炸位置
    explosion_pos = (
        release_position[0] + velocity_vector[0] * fall_time,
        release_position[1] + velocity_vector[1] * fall_time,
        release_position[2] - 0.5 * gravity * fall_time**2,
    )

    return explosion_pos


def calculate_cloud_position(t, explosion_position, explosion_time, sink_speed=3):
    """
    计算特定时刻的云团位置

    参数:
    t -- 当前时间(秒)
    explosion_position -- 爆炸位置 (x,y,z)
    explosion_time -- 爆炸时间(秒)
    sink_speed -- 云团下沉速度(米/秒),默认3

    返回:
    list/None -- 云团位置 [x,y,z] 或 None(未爆炸)
    """
    # 云团匀速下沉
    if t < explosion_time:
        return None  # 未起爆
    else:
        return [
            explosion_position[0],
            explosion_position[1],
            explosion_position[2] - sink_speed * (t - explosion_time),
        ]


def generate_cylinder_sample_points(
    base_center=(0, 200, 0), height=10, radius=7, num_points=5000
):
    """
    生成圆柱体目标上的采样点

    参数:
    base_center -- 底面圆心坐标，默认(0,200,0)
    height -- 圆柱体高度，默认10
    radius -- 圆柱体半径，默认7
    num_points -- 每个圆面上的采样点数，默认100

    返回:
    list -- 圆柱体表面采样点列表
    """
    base_center = np.array(base_center)
    top_center = np.array([base_center[0], base_center[1], base_center[2] + height])

    points = []

    # 底面采样点
    for i in range(num_points):
        angle = 2 * np.pi * i / num_points
        x = base_center[0] + radius * np.cos(angle)
        y = base_center[1] + radius * np.sin(angle)
        z = base_center[2]
        points.append((x, y, z))

    # 顶面采样点
    for i in range(num_points):
        angle = 2 * np.pi * i / num_points
        x = top_center[0] + radius * np.cos(angle)
        y = top_center[1] + radius * np.sin(angle)
        z = top_center[2]
        points.append((x, y, z))

    return points


def calculate_shielding_duration(
    missile_initial_pos=(20000, 0, 2000),
    missile_velocity=300,
    missile_target=(0, 0, 0),
    plane_initial_pos=(17800, 0, 1800),
    plane_target=(0, 0, 1800),
    plane_speed=120,
    release_time=1.5,
    explosion_delay=3.6,
    smoke_radius=10,
    smoke_last_time=20,
    cloud_sink_speed=3,
    target_center=(0, 200, 0),
    target_height=10,
    target_radius=7,
    sample_points=5000,
    time_step=0.005,
    verbose=True,
):
    """
    计算导弹被烟幕有效遮蔽的总时长

    参数:
    missile_initial_pos -- 导弹初始位置，默认(20000,0,2000)
    missile_velocity -- 导弹速度(米/秒)，默认300
    missile_target -- 导弹目标位置(假目标)，默认(0,0,0)
    plane_initial_pos -- 无人机初始位置，默认(17800,0,1800)
    plane_target -- 无人机目标位置，默认(0,0,0)
    plane_speed -- 无人机速度(米/秒)，默认120
    release_time -- 烟幕弹投放时间(秒)，默认1.5
    explosion_delay -- 烟幕弹起爆延迟(秒)，默认3.6
    smoke_radius -- 烟幕云团半径(米)，默认10
    smoke_last_time -- 烟幕有效持续时间(秒)，默认20
    cloud_sink_speed -- 云团下沉速度(米/秒)，默认3
    target_center -- 真目标底面圆心坐标，默认(0,200,0)
    target_height -- 真目标高度(米)，默认10
    target_radius -- 真目标半径(米)，默认7
    sample_points -- 圆柱体每个面的采样点数，默认100
    time_step -- 时间步长(秒)，默认0.01
    verbose -- 是否输出详细信息，默认True

    返回:
    tuple -- (总遮蔽时长(秒), 遮蔽时间段列表[(开始时间,结束时间),...]
    """
    # 计算无人机飞行方向
    plane_direction = np.array(plane_target) - np.array(plane_initial_pos)

    # 计算爆炸位置和时间
    explosion_position = calculate_bomb_position(
        plane_initial_pos, plane_direction, plane_speed, release_time, explosion_delay
    )

    explosion_time = release_time + explosion_delay

    # 生成目标采样点
    target_points = generate_cylinder_sample_points(
        target_center, target_height, target_radius, sample_points
    )

    if verbose:
        print("=== 烟幕干扰分析 ===")
        print(f"无人机初始位置: {plane_initial_pos}")
        print(f"飞行速度: {plane_speed} m/s，朝向目标: {plane_target}")
        print(f"投放时间: {release_time}s")
        print(f"起爆延时: {explosion_delay}s")
        print(f"爆炸时间: {explosion_time}s")
        print(
            f"爆炸位置: ({explosion_position[0]:.1f}, {explosion_position[1]:.1f}, {explosion_position[2]:.1f})"
        )
        print(f"烟幕半径: {smoke_radius}m")
        print(f"烟幕有效时间: {smoke_last_time}s")
        print(f"云团下沉速度: {cloud_sink_speed} m/s")
        print(f"\n目标信息:")
        print(
            f"真目标位置: 圆心{target_center}，半径{target_radius}m，高{target_height}m"
        )
        print(f"采样点数: {len(target_points)}")
        print(f"\n开始模拟...")

    # 遍历时间步长
    time_start = 0
    time_end = 100  # 足够长的时间范围

    shielded_times = []
    is_currently_shielded = False
    shield_start_time = None

    # 记录关键时刻信息
    first_shield_info = None
    last_shield_info = None

    for t in np.arange(time_start, time_end, time_step):
        # 计算当前位置
        missile_pos = calculate_missile_position(
            missile_initial_pos, t, missile_velocity, missile_target
        )
        cloud_pos = calculate_cloud_position(
            t, explosion_position, explosion_time, cloud_sink_speed
        )

        # 检查导弹是否已经通过目标区域
        missile_pos_array = np.array(missile_pos)
        if np.linalg.norm(missile_pos_array) < 1:
            if is_currently_shielded:
                shielded_times.append((shield_start_time, t))
                last_shield_info = {
                    "time": t,
                    "missile_pos": missile_pos,
                    "cloud_pos": cloud_pos,
                    "missile_distance": np.linalg.norm(
                        missile_pos_array - np.array(target_center)
                    ),
                }
            break

        # 检查云团是否已形成
        if cloud_pos is None:
            continue

        # 判断是否有效遮蔽
        is_shielded = is_effectively_shielded(
            missile_pos,
            target_points,
            cloud_pos,
            explosion_time,
            t,
            smoke_radius,
            smoke_last_time,
        )

        # 记录遮蔽时间段
        if is_shielded and not is_currently_shielded:
            is_currently_shielded = True
            shield_start_time = t
            if first_shield_info is None:
                first_shield_info = {
                    "time": t,
                    "missile_pos": missile_pos,
                    "cloud_pos": cloud_pos,
                    "missile_distance": np.linalg.norm(
                        np.array(missile_pos) - np.array(target_center)
                    ),
                }
        elif not is_shielded and is_currently_shielded:
            is_currently_shielded = False
            shielded_times.append((shield_start_time, t))
            last_shield_info = {
                "time": t,
                "missile_pos": missile_pos,
                "cloud_pos": cloud_pos,
                "missile_distance": np.linalg.norm(
                    np.array(missile_pos) - np.array(target_center)
                ),
            }

    # 输出详细信息
    if verbose:
        print(f"\n=== 遮蔽时间段详情 ===")
        if len(shielded_times) == 0:
            print("未检测到有效遮蔽！")
            # 调试信息
            t_test = 10.0  # 测试时刻
            missile_test = calculate_missile_position(
                missile_initial_pos, t_test, missile_velocity, missile_target
            )
            cloud_test = calculate_cloud_position(
                t_test, explosion_position, explosion_time, cloud_sink_speed
            )
            if cloud_test:
                print(f"\n调试信息:")
                print(f"t={t_test}s时:")
                print(f"  导弹位置: {missile_test}")
                print(f"  云团位置: {cloud_test}")
                print(
                    f"  导弹到真目标距离: {np.linalg.norm(np.array(missile_test) - np.array(target_center)):.1f}m"
                )
        else:
            for i, (start, end) in enumerate(shielded_times):
                print(f"第{i+1}段: {start:.2f}s - {end:.2f}s (持续 {end-start:.2f}s)")

            if first_shield_info:
                print(f"\n首次遮蔽时刻 (t={first_shield_info['time']:.2f}s):")
                print(f"  导弹位置: {first_shield_info['missile_pos']}")
                print(f"  云团位置: {first_shield_info['cloud_pos']}")
                print(f"  导弹距真目标: {first_shield_info['missile_distance']:.1f}m")

            if last_shield_info:
                print(f"\n最后遮蔽时刻 (t={last_shield_info['time']:.2f}s):")
                print(f"  导弹位置: {last_shield_info['missile_pos']}")
                print(f"  云团位置: {last_shield_info['cloud_pos']}")
                print(f"  导弹距真目标: {last_shield_info['missile_distance']:.1f}m")

        # 计算总遮蔽时长
        total_duration = sum(end - start for start, end in shielded_times)
        print(f"\n=== 结果汇总 ===")
        print(f"遮蔽段数: {len(shielded_times)}")
        print(f"总遮蔽时长: {total_duration:.4f} 秒")

    return sum(end - start for start, end in shielded_times), shielded_times

In [ ]:
duration, periods = calculate_shielding_duration(
    missile_initial_pos=(20000, 0, 2000),
    missile_velocity=300,
    plane_initial_pos=(17800, 0, 1800),
    plane_target=(210677.516, 20000, 1800),
    plane_speed=119.53,
    release_time=0.36,
    explosion_delay=0.53,
    sample_points=100,  # 减少采样点数以提高性能
)
print(f"问题1的最大遮蔽时长: {duration:.4f}秒")

In [ ]:
import numpy as np
import random
from deap import base, creator, tools, algorithms
import matplotlib.pyplot as plt


def optimize_shielding_parameters():
    # 定义FY1和M1的初始位置
    FY1_initial_pos = (17800, 0, 1800)
    M1_initial_pos = (20000, 0, 2000)

    # 定义参数范围
    # 方位角范围 [-pi, pi]
    # 飞行速度范围 [70, 140]
    # 投放时间范围 [0, 10]
    # 起爆延迟范围 [0, 10]

    # 创建DEAP的类型
    creator.create("FitnessMax", base.Fitness, weights=(1.0,))
    creator.create("Individual", list, fitness=creator.FitnessMax)

    toolbox = base.Toolbox()

    # 定义基因 - 移除了仰角参数
    toolbox.register("attr_azimuth", random.uniform, 0, np.pi / 9)
    toolbox.register("attr_speed", random.uniform, 70, 140)
    toolbox.register("attr_release_time", random.uniform, 0, 1)
    toolbox.register("attr_explosion_delay", random.uniform, 0, 1)

    # 定义个体和种群
    toolbox.register(
        "individual",
        tools.initCycle,
        creator.Individual,
        (
            toolbox.attr_azimuth,
            toolbox.attr_speed,
            toolbox.attr_release_time,
            toolbox.attr_explosion_delay,
        ),
        n=1,
    )
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)

    # 定义评估函数
    def evaluate(individual):
        azimuth, speed, release_time, explosion_delay = individual

        # 将水平方位角转换为笛卡尔坐标系方向向量
        # 确保z方向分量为0（水平飞行）
        direction_x = np.cos(azimuth)
        direction_y = np.sin(azimuth)
        direction_z = 0  # 水平飞行，z分量为0

        plane_direction = (direction_x, direction_y, direction_z)

        # 计算目标位置（保持z坐标不变，确保水平飞行）
        dir_norm = np.sqrt(direction_x**2 + direction_y**2)
        target_distance = 20000  # 足够远的距离
        plane_target = (
            FY1_initial_pos[0] + direction_x / dir_norm * target_distance,
            FY1_initial_pos[1] + direction_y / dir_norm * target_distance,
            FY1_initial_pos[2],  # 保持z坐标不变
        )

        # 使用sample_points=200来加速计算
        try:
            duration, _ = calculate_shielding_duration(
                missile_initial_pos=M1_initial_pos,
                missile_velocity=300,
                missile_target=(0, 0, 0),
                plane_initial_pos=FY1_initial_pos,
                plane_target=plane_target,
                plane_speed=speed,
                release_time=release_time,
                explosion_delay=explosion_delay,
                sample_points=100,
                verbose=False,
            )
            return (duration,)
        except Exception as e:
            print(f"评估出错: {e}")
            return (0.0,)

    # 注册评估、交叉、变异和选择操作
    toolbox.register("evaluate", evaluate)
    toolbox.register("mate", tools.cxBlend, alpha=0.5)
    toolbox.register(
        "mutate", tools.mutGaussian, mu=0, sigma=[0.2, 5, 0.2, 0.2], indpb=0.2
    )
    toolbox.register("select", tools.selTournament, tournsize=3)

    # 在这里插入边界检查代码
    def checkBounds(min_values, max_values):
        def decorator(func):
            def wrapper(*args, **kargs):
                offspring = func(*args, **kargs)
                for child in offspring:
                    for i in range(len(child)):
                        if child[i] > max_values[i]:
                            child[i] = max_values[i]
                        elif child[i] < min_values[i]:
                            child[i] = min_values[i]
                return offspring

            return wrapper

        return decorator

    # 定义参数边界
    MIN_VALUES = [0, 70, 0, 0]  # 最小值
    MAX_VALUES = [np.pi / 9, 140, 1, 1]  # 最大值
    # 应用边界检查
    toolbox.decorate("mate", checkBounds(MIN_VALUES, MAX_VALUES))
    toolbox.decorate("mutate", checkBounds(MIN_VALUES, MAX_VALUES))
    # 设置随机种子以便结果可重现

    # 创建初始种群
    pop_size = 50
    population = toolbox.population(n=pop_size)

    # 记录最佳结果
    hof = tools.HallOfFame(1)

    # 记录统计信息
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("min", np.min)
    stats.register("max", np.max)

    # 运行遗传算法
    generations = 40
    pop, logbook = algorithms.eaSimple(
        population,
        toolbox,
        cxpb=0.7,
        mutpb=0.2,
        ngen=generations,
        stats=stats,
        halloffame=hof,
        verbose=True,
    )

    # 获取最佳个体
    best_ind = hof[0]
    azimuth, speed, release_time, explosion_delay = best_ind

    # 将水平方位角转换为笛卡尔坐标系
    direction_x = np.cos(azimuth)
    direction_y = np.sin(azimuth)
    direction_z = 0  # 水平飞行

    plane_direction = (direction_x, direction_y, direction_z)

    # 计算目标位置（保持高度不变）
    dir_norm = np.sqrt(direction_x**2 + direction_y**2)
    target_distance = 20000
    plane_target = (
        FY1_initial_pos[0] + direction_x / dir_norm * target_distance,
        FY1_initial_pos[1] + direction_y / dir_norm * target_distance,
        FY1_initial_pos[2],  # 保持z坐标不变
    )

    # 计算投放点和起爆点
    release_position = calculate_plane_position(
        FY1_initial_pos, plane_direction, speed, release_time
    )

    explosion_position = calculate_bomb_position(
        FY1_initial_pos, plane_direction, speed, release_time, explosion_delay
    )

    # 使用详细输出验证最佳参数
    best_duration, shield_times = calculate_shielding_duration(
        missile_initial_pos=M1_initial_pos,
        missile_velocity=300,
        missile_target=(0, 0, 0),
        plane_initial_pos=FY1_initial_pos,
        plane_target=plane_target,
        plane_speed=speed,
        release_time=release_time,
        explosion_delay=explosion_delay,
        sample_points=1000,
        verbose=True,
    )

    # 输出最佳参数和结果
    print("\n=== 最佳参数 ===")
    print(f"飞行方向角度: {azimuth * 180 / np.pi:.2f} 度")
    print(f"飞行方向向量: ({direction_x:.4f}, {direction_y:.4f}, {direction_z:.4f})")
    print(f"飞行速度: {speed:.2f} m/s")
    print(f"烟幕干扰弹投放时间: {release_time:.2f} s")
    print(f"烟幕干扰弹起爆延迟: {explosion_delay:.2f} s")
    print(
        f"投放点坐标: ({release_position[0]:.2f}, {release_position[1]:.2f}, {release_position[2]:.2f})"
    )
    print(
        f"起爆点坐标: ({explosion_position[0]:.2f}, {explosion_position[1]:.2f}, {explosion_position[2]:.2f})"
    )
    print(f"最大遮蔽时长: {best_duration:.4f} s")

    # 绘制进化过程图
    gen = range(len(logbook))
    avg = logbook.select("avg")
    max_fit = logbook.select("max")

    plt.figure(figsize=(10, 6))
    plt.plot(gen, avg, label="平均适应度")
    plt.plot(gen, max_fit, label="最大适应度")
    plt.xlabel("代数")
    plt.ylabel("适应度(遮蔽时长)")
    plt.legend()
    plt.title("适应度进化过程")
    plt.grid(True)
    plt.show()

    return (
        best_ind,
        best_duration,
        release_position,
        explosion_position,
        plane_direction,
        speed,
    )

多烟雾的协同

In [ ]:
def is_effectively_shielded_by_multiple_clouds(
    missile_pos,
    target_points,
    cloud_centers,
    explosion_times,
    current_time,
    smoke_radius=10,
    last_time=20,
):
    """
    判断导弹是否被多个烟幕云团有效遮蔽

    参数:
    missile_pos -- 导弹当前位置坐标 (x,y,z)
    target_points -- 目标上的采样点列表
    cloud_centers -- 多个烟幕云团中心位置列表，每个元素为 (x,y,z) 或 None
    explosion_times -- 多个烟幕弹爆炸时间列表
    current_time -- 当前时间
    smoke_radius -- 烟幕云团半径,10m
    last_time -- 烟幕有效持续时间,20s

    返回:
    bool -- 是否被有效遮蔽
    """
    # 筛选有效的云团(已爆炸且未超过持续时间)
    active_clouds = []
    for cloud_center, explosion_time in zip(cloud_centers, explosion_times):
        if cloud_center is not None and current_time - explosion_time <= last_time:
            active_clouds.append(cloud_center)

    # 如果没有有效云团，直接返回False
    if not active_clouds:
        return False

    # 检查每个目标点
    for target_point in target_points:
        # 导弹到目标点的视线向量
        sight_vector = np.array(target_point) - np.array(missile_pos)
        sight_length = np.linalg.norm(sight_vector)

        # 如果导弹已经到达目标点，视为被遮蔽
        if sight_length < 1e-6:
            return True

        # 检查是否至少有一个云团遮蔽了视线
        line_shielded = False

        for cloud_center in active_clouds:
            # 云团中心到视线的投影比例
            mc_vector = np.array(cloud_center) - np.array(missile_pos)
            t = np.dot(mc_vector, sight_vector) / (sight_length**2)

            # 云团中心到视线的垂直距离
            distance = np.linalg.norm(mc_vector - t * sight_vector)

            # 云团是否在导弹和目标之间
            in_between = 0 <= t <= 1

            # 判断该云团是否遮蔽了视线
            if distance < smoke_radius and in_between:
                line_shielded = True
                break
            elif distance < smoke_radius:
                # 如果云团不在导弹和目标之间，检查是否足够接近目标或导弹
                if t > 1:  # 云团在目标"后方"
                    distance_to_target = np.linalg.norm(
                        np.array(cloud_center) - np.array(target_point)
                    )
                    if distance_to_target < smoke_radius:
                        line_shielded = True
                        break
                else:  # 云团在导弹"后方"(t<0)
                    distance_to_missile = np.linalg.norm(
                        np.array(cloud_center) - np.array(missile_pos)
                    )
                    if distance_to_missile < smoke_radius:
                        line_shielded = True
                        break

        # 如果所有云团都未能遮蔽该视线，则整体未被遮蔽
        if not line_shielded:
            return False

    # 所有视线都被遮蔽，则整体被遮蔽
    return True


def calculate_multi_bombs_shielding_duration(
    missile_initial_pos=(20000, 0, 2000),
    missile_velocity=300,
    missile_target=(0, 0, 0),
    plane_initial_pos=(17800, 0, 1800),
    plane_target=(0, 0, 1800),
    plane_speed=120,
    release_times=[1.5, 5.0, 10.0],
    explosion_delays=[3.6, 3.6, 3.6],
    smoke_radius=10,
    smoke_last_time=20,
    cloud_sink_speed=3,
    target_center=(0, 200, 0),
    target_height=10,
    target_radius=7,
    sample_points=100,
    time_step=0.05,
    verbose=False,
):
    """
    计算三枚烟幕弹协同作用下导弹被有效遮蔽的总时长

    参数:
    missile_initial_pos -- 导弹初始位置
    missile_velocity -- 导弹速度(米/秒)
    missile_target -- 导弹目标位置(假目标)
    plane_initial_pos -- 无人机初始位置
    plane_target -- 无人机目标位置
    plane_speed -- 无人机速度(米/秒)
    release_times -- 三枚烟幕弹的投放时间列表
    explosion_delays -- 三枚烟幕弹的起爆延迟列表
    其他参数与单枚烟幕弹相同

    返回:
    tuple -- (总遮蔽时长(秒), 遮蔽时间段列表[(开始时间,结束时间),...]
    """
    # 计算无人机飞行方向
    plane_direction = np.array(plane_target) - np.array(plane_initial_pos)
    plane_direction = plane_direction / np.linalg.norm(plane_direction)

    # 计算每枚烟幕弹的爆炸位置和时间
    explosion_positions = []
    explosion_times = []

    for release_time, explosion_delay in zip(release_times, explosion_delays):
        explosion_pos = calculate_bomb_position(
            plane_initial_pos,
            plane_direction,
            plane_speed,
            release_time,
            explosion_delay,
        )
        explosion_time = release_time + explosion_delay

        explosion_positions.append(explosion_pos)
        explosion_times.append(explosion_time)

    # 生成目标采样点
    target_points = generate_cylinder_sample_points(
        target_center, target_height, target_radius, sample_points
    )

    if verbose:
        print("=== 三枚烟幕弹协同干扰分析 ===")
        print(f"无人机初始位置: {plane_initial_pos}")
        print(f"飞行速度: {plane_speed} m/s，朝向目标: {plane_target}")
        for i, (rt, ed, ep, et) in enumerate(
            zip(release_times, explosion_delays, explosion_positions, explosion_times)
        ):
            print(f"\n烟幕弹 {i+1}:")
            print(f"  投放时间: {rt:.2f}s")
            print(f"  起爆延时: {ed:.2f}s")
            print(f"  爆炸时间: {et:.2f}s")
            print(f"  爆炸位置: ({ep[0]:.1f}, {ep[1]:.1f}, {ep[2]:.1f})")

        print(f"\n烟幕参数:")
        print(f"  烟幕半径: {smoke_radius}m")
        print(f"  烟幕有效时间: {smoke_last_time}s")
        print(f"  云团下沉速度: {cloud_sink_speed} m/s")
        print(f"\n目标信息:")
        print(
            f"  真目标位置: 圆心{target_center}，半径{target_radius}m，高{target_height}m"
        )
        print(f"  采样点数: {len(target_points)}")
        print(f"\n开始模拟协同遮蔽效果...")

    # 遍历时间步长
    time_start = 0
    time_end = 100  # 足够长的时间范围

    shielded_times = []
    is_currently_shielded = False
    shield_start_time = None

    # 记录关键时刻信息
    first_shield_info = None
    last_shield_info = None

    for t in np.arange(time_start, time_end, time_step):
        # 计算当前导弹位置
        missile_pos = calculate_missile_position(
            missile_initial_pos, t, missile_velocity, missile_target
        )

        # 检查导弹是否已经通过目标区域
        missile_pos_array = np.array(missile_pos)
        if np.linalg.norm(missile_pos_array) < 1:
            if is_currently_shielded:
                shielded_times.append((shield_start_time, t))
                last_shield_info = {
                    "time": t,
                    "missile_pos": missile_pos,
                    "missile_distance": np.linalg.norm(
                        missile_pos_array - np.array(target_center)
                    ),
                }
            break

        # 计算当前各云团位置
        cloud_positions = []
        for explosion_pos, explosion_time in zip(explosion_positions, explosion_times):
            cloud_pos = calculate_cloud_position(
                t, explosion_pos, explosion_time, cloud_sink_speed
            )
            cloud_positions.append(cloud_pos)

        # 判断是否有效遮蔽（考虑多云团协同）
        is_shielded = is_effectively_shielded_by_multiple_clouds(
            missile_pos,
            target_points,
            cloud_positions,
            explosion_times,
            t,
            smoke_radius,
            smoke_last_time,
        )

        # 记录遮蔽时间段
        if is_shielded and not is_currently_shielded:
            is_currently_shielded = True
            shield_start_time = t
            if first_shield_info is None:
                first_shield_info = {
                    "time": t,
                    "missile_pos": missile_pos,
                    "cloud_positions": [cp for cp in cloud_positions if cp is not None],
                    "missile_distance": np.linalg.norm(
                        np.array(missile_pos) - np.array(target_center)
                    ),
                }
        elif not is_shielded and is_currently_shielded:
            is_currently_shielded = False
            shielded_times.append((shield_start_time, t))
            last_shield_info = {
                "time": t,
                "missile_pos": missile_pos,
                "cloud_positions": [cp for cp in cloud_positions if cp is not None],
                "missile_distance": np.linalg.norm(
                    np.array(missile_pos) - np.array(target_center)
                ),
            }

    # 输出详细信息
    if verbose:
        print(f"\n=== 协同遮蔽时间段详情 ===")
        if len(shielded_times) == 0:
            print("未检测到有效遮蔽！")
        else:
            for i, (start, end) in enumerate(shielded_times):
                print(f"第{i+1}段: {start:.2f}s - {end:.2f}s (持续 {end-start:.2f}s)")

            if first_shield_info:
                print(f"\n首次遮蔽时刻 (t={first_shield_info['time']:.2f}s):")
                print(f"  导弹位置: {first_shield_info['missile_pos']}")
                print(f"  导弹距真目标: {first_shield_info['missile_distance']:.1f}m")
                print(f"  活跃云团数: {len(first_shield_info['cloud_positions'])}")

            if last_shield_info:
                print(f"\n最后遮蔽时刻 (t={last_shield_info['time']:.2f}s):")
                print(f"  导弹位置: {last_shield_info['missile_pos']}")
                print(f"  导弹距真目标: {last_shield_info['missile_distance']:.1f}m")
                print(f"  活跃云团数: {len(last_shield_info['cloud_positions'])}")

        # 计算总遮蔽时长
        total_duration = sum(end - start for start, end in shielded_times)
        print(f"\n=== 结果汇总 ===")
        print(f"遮蔽段数: {len(shielded_times)}")
        print(f"总遮蔽时长: {total_duration:.4f} 秒")

    return sum(end - start for start, end in shielded_times), shielded_times

In [ ]:
import numpy as np
import random
from deap import base, creator, tools, algorithms
import matplotlib.pyplot as plt
import pandas as pd


def optimize_three_bombs_strategy():
    """优化三枚烟幕弹的协同投放策略"""
    # 重置DEAP的创建器（如果之前已经定义过）
    if hasattr(creator, "FitnessMax"):
        del creator.FitnessMax
    if hasattr(creator, "Individual"):
        del creator.Individual

    # 定义FY1和M1的初始位置
    FY1_initial_pos = (17800, 0, 1800)
    M1_initial_pos = (20000, 0, 2000)

    # 创建DEAP的类型
    creator.create("FitnessMax", base.Fitness, weights=(1.0,))
    creator.create("Individual", list, fitness=creator.FitnessMax)

    toolbox = base.Toolbox()

    # 定义基因 - 共8个参数
    toolbox.register("attr_azimuth", random.uniform, 0, np.pi / 10)  # 方位角
    toolbox.register("attr_speed", random.uniform, 90, 120)  # 飞行速度
    toolbox.register("attr_release_time1", random.uniform, 0, 1)  # 第一枚投放时间
    toolbox.register("attr_release_time2", random.uniform, 0, 10)  # 第二枚投放时间
    toolbox.register("attr_release_time3", random.uniform, 7, 15)  # 第三枚投放时间
    toolbox.register("attr_explosion_delay1", random.uniform, 0, 1)  # 第一枚延迟
    toolbox.register("attr_explosion_delay2", random.uniform, 0, 1)  # 第二枚延迟
    toolbox.register("attr_explosion_delay3", random.uniform, 0, 1)  # 第三枚延迟

    # 定义个体和种群
    toolbox.register(
        "individual",
        tools.initCycle,
        creator.Individual,
        (
            toolbox.attr_azimuth,
            toolbox.attr_speed,
            toolbox.attr_release_time1,
            toolbox.attr_explosion_delay1,
            toolbox.attr_release_time2,
            toolbox.attr_explosion_delay2,
            toolbox.attr_release_time3,
            toolbox.attr_explosion_delay3,
        ),
        n=1,
    )
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)

    # 定义评估函数
    def evaluate(individual):
        # 解析参数
        (
            azimuth,
            speed,
            release_time1,
            explosion_delay1,
            release_time2,
            explosion_delay2,
            release_time3,
            explosion_delay3,
        ) = individual

        # 计算飞行方向向量
        direction_x = np.cos(azimuth)
        direction_y = np.sin(azimuth)
        direction_z = 0  # 水平飞行

        plane_direction = (direction_x, direction_y, direction_z)

        # 计算目标位置（保持z坐标不变）
        dir_norm = np.sqrt(direction_x**2 + direction_y**2)
        target_distance = 20000  # 足够远的距离
        plane_target = (
            FY1_initial_pos[0] + direction_x / dir_norm * target_distance,
            FY1_initial_pos[1] + direction_y / dir_norm * target_distance,
            FY1_initial_pos[2],  # 保持z坐标不变
        )

        # 计算三枚烟幕弹的遮蔽区间
        total_shielding_periods = []

        # 三枚烟幕弹的计算
        for release_time, explosion_delay in [
            (release_time1, explosion_delay1),
            (release_time2, explosion_delay2),
            (release_time3, explosion_delay3),
        ]:
            try:
                _, shield_times = calculate_shielding_duration(
                    missile_initial_pos=M1_initial_pos,
                    missile_velocity=300,
                    missile_target=(0, 0, 0),
                    plane_initial_pos=FY1_initial_pos,
                    plane_target=plane_target,
                    plane_speed=speed,
                    release_time=release_time,
                    explosion_delay=explosion_delay,
                    sample_points=200,  # 减少采样点以提高速度
                    time_step=0.01,  # 增大时间步长以提高速度
                    verbose=False,
                )
                total_shielding_periods.extend(shield_times)
            except Exception as e:
                pass

        # 如果没有有效遮蔽，返回0
        if not total_shielding_periods:
            return (0.0,)

        # 合并重叠的遮蔽时间段
        total_shielding_periods.sort(key=lambda x: x[0])
        merged_periods = [total_shielding_periods[0]]

        for current in total_shielding_periods[1:]:
            previous = merged_periods[-1]
            if current[0] <= previous[1]:
                # 有重叠，合并区间
                merged_periods[-1] = (previous[0], max(previous[1], current[1]))
            else:
                # 无重叠，添加新区间
                merged_periods.append(current)

        # 计算总遮蔽时长
        total_duration = sum(end - start for start, end in merged_periods)
        return (total_duration,)

    # 注册评估函数和遗传操作
    toolbox.register("evaluate", evaluate)
    toolbox.register("mate", tools.cxBlend, alpha=0.5)
    toolbox.register(
        "mutate",
        tools.mutGaussian,
        mu=0,
        sigma=[0.2, 5, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0],  # 调整变异幅度
        indpb=0.2,
    )
    toolbox.register("select", tools.selTournament, tournsize=3)

    # 边界检查函数
    def checkBounds():
        """
        边界检查函数 - 确保所有参数都在合理范围内
        """

        def decorator(func):
            def wrapper(*args, **kargs):
                offspring = func(*args, **kargs)

                for child in offspring:
                    # 1. 方位角约束 [0, π/15]
                    # 想象这是无人机的飞行方向角度，不能超出预设范围
                    if child[0] > np.pi / 10:
                        child[0] = np.pi / 10
                    elif child[0] < 0:
                        child[0] = 0

                    # 2. 飞行速度约束 [90, 120] m/s
                    # 这是无人机的物理性能限制
                    if child[1] > 120:
                        child[1] = 120
                    elif child[1] < 90:
                        child[1] = 90

                    # 3. 第一枚烟幕弹投放时间约束 [0, 1] s
                    if child[2] > 1:
                        child[2] = 1
                    elif child[2] < 0:
                        child[2] = 0

                    # 4. 第一枚烟幕弹延迟时间约束 [0, 0.5] s
                    if child[3] > 1:
                        child[3] = 1
                    elif child[3] < 0:
                        child[3] = 0

                    # 5. 第二枚烟幕弹投放时间约束 [0, 5] s
                    if child[4] > 10:
                        child[4] = 10
                    elif child[4] < 0:
                        child[4] = 0

                    # 6. 第二枚烟幕弹延迟时间约束 [0, 0.5] s
                    if child[5] > 1:
                        child[5] = 1
                    elif child[5] < 0:
                        child[5] = 0

                    # 7. 第三枚烟幕弹投放时间约束 [7, 15] s
                    if child[6] > 15:
                        child[6] = 15
                    elif child[6] < 7:
                        child[6] = 7

                    # 8. 第三枚烟幕弹延迟时间约束 [0, 0.5] s
                    if child[7] > 1:
                        child[7] = 1
                    elif child[7] < 0:
                        child[7] = 0

                    if child[4] <= child[2] + 1:  # 第二枚必须比第一枚晚至少1秒
                        child[4] = child[2] + 1
                    if child[6] <= child[4] + 1:  # 第三枚必须比第二枚晚至少1秒
                        child[6] = child[4] + 1
                return offspring

            return wrapper

        return decorator

    # 应用边界检查
    toolbox.decorate("mate", checkBounds())
    toolbox.decorate("mutate", checkBounds())

    # 创建初始种群（减小种群规模以补偿单进程的速度劣势）
    pop_size = 70  # 从100减少到50
    population = toolbox.population(n=pop_size)

    # 记录最佳结果
    hof = tools.HallOfFame(1)

    # 记录统计信息
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("min", np.min)
    stats.register("max", np.max)

    # 运行遗传算法（减少代数以节省时间）
    generations = 100  # 从50减少到30
    pop, logbook = algorithms.eaSimple(
        population,
        toolbox,
        cxpb=0.6,  # 交叉概率
        mutpb=0.3,  # 变异概率
        ngen=generations,
        stats=stats,
        halloffame=hof,
        verbose=True,
    )

    # 获取最佳个体
    best_ind = hof[0]

    # 解析最佳参数
    (
        azimuth,
        speed,
        release_time1,
        explosion_delay1,
        release_time2,
        explosion_delay2,
        release_time3,
        explosion_delay3,
    ) = best_ind

    # 计算飞行方向
    direction_x = np.cos(azimuth)
    direction_y = np.sin(azimuth)
    direction_z = 0
    direction = (direction_x, direction_y, direction_z)

    # 计算目标位置
    dir_norm = np.sqrt(direction_x**2 + direction_y**2)
    target_distance = 20000
    plane_target = (
        FY1_initial_pos[0] + direction_x / dir_norm * target_distance,
        FY1_initial_pos[1] + direction_y / dir_norm * target_distance,
        FY1_initial_pos[2],
    )

    # 转换为角度（0-360度）
    direction_angle = (np.arctan2(direction_y, direction_x) * 180 / np.pi) % 360

    # 计算三枚烟幕弹的详细参数
    bombs_data = []
    for idx, (release_time, explosion_delay) in enumerate(
        [
            (release_time1, explosion_delay1),
            (release_time2, explosion_delay2),
            (release_time3, explosion_delay3),
        ]
    ):

        # 计算投放点和起爆点
        release_position = calculate_plane_position(
            FY1_initial_pos, direction, speed, release_time
        )

        explosion_position = calculate_bomb_position(
            FY1_initial_pos, direction, speed, release_time, explosion_delay
        )

        # 计算单枚干扰时长
        duration, _ = calculate_shielding_duration(
            missile_initial_pos=M1_initial_pos,
            missile_velocity=300,
            missile_target=(0, 0, 0),
            plane_initial_pos=FY1_initial_pos,
            plane_target=plane_target,
            plane_speed=speed,
            release_time=release_time,
            explosion_delay=explosion_delay,
            sample_points=300,
            verbose=False,
        )

        bombs_data.append(
            {
                "编号": idx + 1,
                "运动方向": direction_angle,
                "运动速度": speed,
                "投放点x": release_position[0],
                "投放点y": release_position[1],
                "投放点z": release_position[2],
                "起爆点x": explosion_position[0],
                "起爆点y": explosion_position[1],
                "起爆点z": explosion_position[2],
                "有效干扰时长": duration,
            }
        )

    # 计算合并后的总有效干扰时长
    total_shielding_periods = []
    for release_time, explosion_delay in [
        (release_time1, explosion_delay1),
        (release_time2, explosion_delay2),
        (release_time3, explosion_delay3),
    ]:
        _, shield_times = calculate_shielding_duration(
            missile_initial_pos=M1_initial_pos,
            missile_velocity=300,
            missile_target=(0, 0, 0),
            plane_initial_pos=FY1_initial_pos,
            plane_target=plane_target,
            plane_speed=speed,
            release_time=release_time,
            explosion_delay=explosion_delay,
            sample_points=300,
            verbose=False,
        )
        total_shielding_periods.extend(shield_times)

    # 合并重叠的遮蔽时间段
    if total_shielding_periods:
        total_shielding_periods.sort(key=lambda x: x[0])
        merged_periods = [total_shielding_periods[0]]

        for current in total_shielding_periods[1:]:
            previous = merged_periods[-1]
            if current[0] <= previous[1]:
                # 有重叠，合并区间
                merged_periods[-1] = (previous[0], max(previous[1], current[1]))
            else:
                # 无重叠，添加新区间
                merged_periods.append(current)

        total_duration = sum(end - start for start, end in merged_periods)
    else:
        total_duration = 0.0
        merged_periods = []

    # 创建结果DataFrame
    df = pd.DataFrame(
        columns=[
            "无人机运动方向",
            "无人机运动速度 (m/s)",
            "烟幕干扰弹编号",
            "烟幕干扰弹投放点的x坐标 (m)",
            "烟幕干扰弹投放点的y坐标 (m)",
            "烟幕干扰弹投放点的z坐标 (m)",
            "烟幕干扰弹起爆点的x坐标 (m)",
            "烟幕干扰弹起爆点的y坐标 (m)",
            "烟幕干扰弹起爆点的z坐标 (m)",
            "有效干扰时长 (s)",
        ]
    )

    # 打印优化结果
    print("\n=== 优化结果 ===")
    print(f"无人机运动方向: {direction_angle:.2f}度")
    print(f"无人机运动速度: {speed:.2f} m/s")
    print(f"总有效干扰时长: {total_duration:.2f}秒")
    print(
        f"烟幕弹1: 投放时间={release_time1:.2f}s, 延迟={explosion_delay1:.2f}s, 干扰时长={bombs_data[0]['有效干扰时长']:.2f}s"
    )
    print(
        f"烟幕弹2: 投放时间={release_time2:.2f}s, 延迟={explosion_delay2:.2f}s, 干扰时长={bombs_data[1]['有效干扰时长']:.2f}s"
    )
    print(
        f"烟幕弹3: 投放时间={release_time3:.2f}s, 延迟={explosion_delay3:.2f}s, 干扰时长={bombs_data[2]['有效干扰时长']:.2f}s"
    )

    return best_ind, bombs_data, total_duration, merged_periods


# 运行优化
if __name__ == "__main__":
    best_params, bombs_data, total_duration, shield_periods = (
        optimize_three_bombs_strategy()
    )
    print(f"优化完成，最大总遮蔽时长: {total_duration:.2f}秒")